# App data generation
This notebook is used to serialise waveform and model output data into .pkl format for use by the dashboard

In [1]:
import seisbench.data as sbd
import seisbench.util as sbu
import seisbench.generate as sbg
import seisbench.models as sbm
from seisbench.util import worker_seeding

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from obspy import UTCDateTime

import os, obspy, sys, pickle
from pathlib import Path
from tqdm import tqdm
import pandas as pd


PROJ_ROOT = 'c:\\Users\\deqia\\Documents\\University of Helsinki\\HY 2026 Period 4\\Period 3 DATA11004 Data Science Project I\\eqcomp'

# RUN THIS IS PROJECT ROOT I.E. eqcomp/
os.chdir(PROJ_ROOT)

import utils.config as cfg
import utils.eval as eutils
import utils.model as mutils
import utils.train as tutils
import vis.app_utils as autils


dir_data = cfg.DIR_VIS / autils.DIR_VIS_DATA

# Load pre-trained model from seisbench Model weights are in fp32 (float)
configs = {"EQTransformer": mutils.EQTransformerConfig(),
           "EQCCTP": mutils.EQCCTConfig("P"),
           "EQCCTS": mutils.EQCCTConfig("S"),
           "PhaseNet": mutils.PhaseNetConfig()
        } # Add here

In [2]:
def build_sample_object(sample, metadata, models, model_names, threshold=0.2):
    """
    Build a unified sample object containing:
    - waveform
    - ground truth labels
    - true picks
    - predictions from multiple models
    - predicted picks
    - errors (in seconds)
    - TP/FP/FN/TN classification
    - merged EQCCT (EQCCTP + EQCCTS)
    """

    # 1. Extract waveform + ground truth
    X = sample["X"]              # shape (3, T)
    y_true = sample["y"]         # shape (3, T)

    # 2. Extract true picks from metadata
    true_picks = {
        "P": metadata.get("trace_p_arrival_sample", None),
        "S": metadata.get("trace_s_arrival_sample", None),
    }

    # 3. Extract event type
    event_type = metadata.get("event_type", "unknown")

    # 4. Noise → override true picks
    if event_type == "noise":
        true_picks = {"P": None, "S": None}

    # Unified sample object
    obj = {
        "index": metadata.get("index", None),
        "X": X,
        "y_true": y_true,
        "true_picks": true_picks,
        "event_type": event_type,
        "source_id": metadata.get("source_id", None),
        "source_origin_time": str(metadata.get("source_origin_time", None)),
        "source_depth_km": metadata.get("source_depth_km", None),
        "source_magnitude": metadata.get("source_magnitude", None),
        "station_network_code": metadata.get("station_network_code", None),
        "station_code": metadata.get("station_code", None),
        "trace_channel": metadata.get("trace_channel", None),
        "trace_start_time": metadata.get("trace_start_time", None),
        "predictions": {},
    }
    
    # Prepare merged EQCCT containers
    eqcct_probs = {}
    eqcct_picks = {}
    eqcct_errors = {}
    eqcct_tp_fp_fn = {}


    for model, name in zip(models, model_names):
        raw_probs = eutils.run_model(model, sample)  # shape (C, T)
        probs = dict(zip(model.labels, raw_probs))

        model_labels = list(model.labels)
        model_phases = [phase for phase in ["P", "S"] if phase in model_labels]
        
        pred_picks = {}
        for phase in model_phases:
            pred_picks[phase] = eutils.get_predicted_pick(probs[phase])

        errors = {}
        tp_fp_fn = {}

        for phase in model_phases:
            pred_pick = pred_picks[phase]
            true_pick = true_picks.get(phase, None)
            # print(f"phase {phase}:  pred: {pred_pick}  true_pick:{true_pick} ")

            # No true picks for a particular phase
            if true_pick is None or not np.isfinite(true_pick):
                errors[phase] = None
                tp_fp_fn[phase] = "TN" if pred_pick is None else "FP"
                continue

            # Missed pick
            if pred_pick is None or not np.isfinite(pred_pick):
                errors[phase] = None
                tp_fp_fn[phase] = "FN"
                continue

            # Compute signed error in seconds
            error = (pred_pick - true_pick) / cfg.SAMPLING_RATE
            errors[phase] = error

            # Threshold determines TP vs FP
            tp_fp_fn[phase] = "TP" if abs(error) <= threshold else "FP"
            # print(tp_fp_fn[phase])

        # MERGE EQCCTP and EQCCTS to EQCCT
        if name in ["EQCCTP", "EQCCTS"]:
            for phase in model_phases:
                eqcct_probs[phase] = probs[phase]
                eqcct_picks[phase] = pred_picks[phase]
                eqcct_errors[phase] = errors[phase]
                eqcct_tp_fp_fn[phase] = tp_fp_fn[phase]

        else:
            # Normal model
            obj["predictions"][name] = {
                "probs": probs,
                "picks": pred_picks,
                "errors": errors,
                "tp_fp_fn": tp_fp_fn,
            }

    # Store merged EQCCT
    if len(eqcct_probs) > 0:
        obj["predictions"]["EQCCT"] = {
            "probs": eqcct_probs,
            "picks": eqcct_picks,
            "errors": eqcct_errors,
            "tp_fp_fn": eqcct_tp_fp_fn,
        }

    return obj


def build_all_samples(generator, dataset, models, model_names, limit=None):
    """
    Build unified sample objects for an entire dataset.

    Parameters
    ----------
    dataset : list or Dataset-like
        Must support __len__ and __getitem__.
    models : list
        List of model instances.
    model_names : list
        List of model names corresponding to each model.
    limit : int or None
        Optional limit on number of samples to process.

    Returns
    -------
    list
        List of unified sample objects.
    """
    unified = []
    N = len(generator) if limit is None else min(limit, len(generator))

    for i in tqdm(range(N)):
        metadata = dataset.get_sample(i)[1]
        sample = generator[i]
        obj = build_sample_object(sample, metadata, models, model_names)
        unified.append(obj)

    return unified

In [3]:
# Load the train validation and test splits
dataset = sbd.WaveformDataset(cfg.DIR_DATA)
train_dataset, dev_dataset, test_dataset = dataset.train_dev_test()

# Preprocess dataset
train_gen, dev_gen, test_gen = tutils.preprocess_data(mutils.EQTransformerConfig(), train_dataset, dev_dataset, test_dataset)

# Preview streamlit unified object output
i = 303
sample = dev_gen[i]
sample_meta = dev_dataset.get_sample(i)[1]
models = [config.get_new_model() for config in configs.values()]
model_names = configs.keys()
obj = build_sample_object(sample, sample_meta, models, model_names)
# obj["predictions"]
obj

2026-09-06 15:47:55,410 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


{'index': 3172,
 'X': array([[ -0.        , -13.471004  ,  -6.312393  , ...,   1.450396  ,
           0.5647936 ,   0.        ],
        [ -0.        ,  -4.5411024 ,  -1.7687621 , ...,   0.38934156,
           0.4280385 ,   0.        ],
        [ -0.        ,  -0.7693549 ,  -1.6660997 , ...,  -0.85033005,
          -0.5225073 ,  -0.        ]], shape=(3, 6000), dtype=float32),
 'y_true': array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(3, 6000), dtype=float32),
 'true_picks': {'P': 2217.0, 'S': nan},
 'event_type': 'explosions',
 'source_id': 'smi:fi.isuh/event/1491054',
 'source_origin_time': '2025-01-05 00:31:37.900000+00:00',
 'source_depth_km': 0.0,
 'source_magnitude': 1.1,
 'station_network_code': 'FN',
 'station_code': 'RNF',
 'trace_channel': 'HH',
 'trace_start_time': '2025-01-05T00:32:08.430000Z',
 'predictions': {'EQTransformer': {'probs': {'Detection': array([6.0943435e-03, 2.1999856e-03, 7.6426653e-0

In [4]:
# Build unified sample objects
unified_samples = build_all_samples(
    generator=dev_gen,          # or dev/test
    dataset = dev_dataset,
    models=[config.get_new_model() for config in configs.values()],
    model_names=configs.keys(),
    limit=None                      # or limit=500 for speed
)

def split_pickle(unified_samples):
    total = len(unified_samples)
    num_parts = (total + autils.CHUNK_SIZE - 1) // autils.CHUNK_SIZE

    for i in tqdm(range(num_parts)):
        start = i * autils.CHUNK_SIZE
        end = min(start + autils.CHUNK_SIZE, total)
        chunk = unified_samples[start:end]

        filename = dir_data / f"{autils.FILENAME_PREFIX}_{i}.pkl"
        with open(filename, "wb") as f:
            pickle.dump(chunk, f)

# Run the split
split_pickle(unified_samples)

100%|██████████| 8/8 [00:02<00:00,  3.45it/s]


In [5]:
def load_all_samples():
    samples = []
    for pkl_file in sorted(dir_data.glob(f"{autils.FILENAME_PREFIX}_*.pkl")):
        with open(pkl_file, "rb") as f:
            part = pickle.load(f)
            samples.extend(part)
    return samples

samples = load_all_samples()

In [6]:
samples

[{'index': 36,
  'X': array([[ 0.        ,  0.5275826 ,  0.73586106, ...,  6.2998133 ,
           1.993171  ,  0.        ],
         [-0.        , -0.3178691 , -0.4334356 , ..., -4.5420675 ,
          -0.82312363, -0.        ],
         [ 0.        , -0.02940794,  0.15856154, ...,  3.6553166 ,
          -0.8650203 , -0.        ]], shape=(3, 6000), dtype=float32),
  'y_true': array([[0., 0., 0., ..., 1., 1., 1.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]], shape=(3, 6000), dtype=float32),
  'true_picks': {'P': 4040.355000000001, 'S': 5189.086},
  'event_type': 'earthquakes',
  'source_id': 'smi:fi.isuh/event/1491278',
  'source_origin_time': '2025-01-07 04:14:16.900000+00:00',
  'source_depth_km': 16.3,
  'source_magnitude': 0.8,
  'station_network_code': 'FN',
  'station_code': 'OUL',
  'trace_channel': 'HH',
  'trace_start_time': '2025-01-07T04:13:52.140000Z',
  'predictions': {'EQTransformer': {'probs': {'Detection': array([0.00793304, 0.00306664,